In [1]:
import pandas as pd, sqlite3, os

df = pd.read_csv("data-processed/order_items_clean.csv", parse_dates=["order_date", "shipping_date"])
print(df.shape)

(180519, 46)


In [2]:
def build_dim(df, key, cols):
    dim = df[[key] + cols].drop_duplicates()
    status = "OK" if dim[key].is_unique else "NOT UNIQUE"
    print(f"{key}: {dim[key].nunique()} keys, {len(dim)} rows -> {status}")
    return dim.sort_values(key).reset_index(drop=True)

dim_customer = build_dim(df, "customer_id", ["customer_segment", "customer_city", "customer_state", "customer_country"])
dim_product = build_dim(df, "product_card_id", ["product_name", "product_price", "category_id", "category_name",
                                                 "department_id", "department_name"])

customer_id: 20652 keys, 20652 rows -> OK
product_card_id: 118 keys, 118 rows -> OK


In [3]:
loc_cols = ["market", "order_region", "order_country", "order_state", "order_city"]
dim_location = df[loc_cols].drop_duplicates().sort_values(loc_cols).reset_index(drop=True)
dim_location.insert(0, "location_id", range(1, len(dim_location) + 1))

dim_shipping_mode = pd.DataFrame({"shipping_mode": ["Same Day", "First Class", "Second Class", "Standard Class"]})
dim_shipping_mode.insert(0, "shipping_mode_id", range(1, 5))

print("locations:", len(dim_location))
dim_shipping_mode

locations: 3772


,shipping_mode_id,shipping_mode
0,1,Same Day
1,2,First Class
2,3,Second Class
3,4,Standard Class


In [4]:
dates = pd.date_range(df["order_date"].min().normalize(), df["shipping_date"].max().normalize(), freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["week_start"] = (dim_date["date"] - pd.to_timedelta(dim_date["date"].dt.weekday, unit="D")).dt.date
dim_date["day_name"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = dim_date["date"].dt.weekday >= 5
dim_date["in_analysis_period"] = dim_date["date"] < "2017-10-01"
dim_date["date"] = dim_date["date"].dt.date
dim_date = dim_date[["date_key", "date", "year", "quarter", "month", "month_name", "week_start",
                     "day_name", "is_weekend", "in_analysis_period"]]
print(len(dim_date), "days")
dim_date.head()

1133 days


,date_key,date,year,quarter,month,month_name,week_start,day_name,is_weekend,in_analysis_period
0,20150101,2015-01-01,2015,1,1,Jan,2014-12-29,Thursday,False,True
1,20150102,2015-01-02,2015,1,1,Jan,2014-12-29,Friday,False,True
2,20150103,2015-01-03,2015,1,1,Jan,2014-12-29,Saturday,True,True
3,20150104,2015-01-04,2015,1,1,Jan,2014-12-29,Sunday,True,True
4,20150105,2015-01-05,2015,1,1,Jan,2015-01-05,Monday,False,True


In [5]:
fact = (df.merge(dim_location, on=loc_cols, how="left")
          .merge(dim_shipping_mode, on="shipping_mode", how="left"))
fact["order_date_key"] = fact["order_date"].dt.strftime("%Y%m%d").astype(int)
fact["shipping_date_key"] = fact["shipping_date"].dt.strftime("%Y%m%d").astype(int)

fact = fact[["order_item_id", "order_id", "order_date_key", "shipping_date_key", "customer_id", "product_card_id",
             "location_id", "shipping_mode_id", "payment_type", "order_status", "delivery_status",
             "is_cancelled", "is_late", "late_delivery_risk", "days_for_shipping_real",
             "days_for_shipment_scheduled", "delay_days", "order_item_quantity", "order_item_product_price",
             "order_item_discount", "order_item_discount_rate", "sales", "order_item_total",
             "order_profit_per_order"]].rename(columns={"order_profit_per_order": "profit"})
print(fact.shape)

(180519, 24)


In [6]:
model_checks = {
    "fact rows = clean rows": len(fact) == len(df),
    "customer_id = order_customer_id": (df["customer_id"] == df["order_customer_id"]).all(),
    "every customer exists in dim_customer": fact["customer_id"].isin(dim_customer["customer_id"]).all(),
    "every product exists in dim_product": fact["product_card_id"].isin(dim_product["product_card_id"]).all(),
    "every row has a location_id": fact["location_id"].notna().all(),
    "every row has a shipping_mode_id": fact["shipping_mode_id"].notna().all(),
    "every order date exists in dim_date": fact["order_date_key"].isin(dim_date["date_key"]).all(),
    "every shipping date exists in dim_date": fact["shipping_date_key"].isin(dim_date["date_key"]).all(),
    "total sales unchanged": abs(fact["sales"].sum() - df["sales"].sum()) < 0.01,
}
pd.Series(model_checks).map({True: "PASS", False: "FAIL"})

fact rows = clean rows                    PASS
customer_id = order_customer_id           PASS
every customer exists in dim_customer     PASS
every product exists in dim_product       PASS
every row has a location_id               PASS
every row has a shipping_mode_id          PASS
every order date exists in dim_date       PASS
every shipping date exists in dim_date    PASS
total sales unchanged                     PASS
dtype: object

In [7]:
tables = {"fact_order_items": fact, "dim_customer": dim_customer, "dim_product": dim_product,
          "dim_location": dim_location, "dim_shipping_mode": dim_shipping_mode, "dim_date": dim_date}

os.makedirs("model", exist_ok=True)
con = sqlite3.connect("supply_chain.db")
for name, table in tables.items():
    table.to_sql(name, con, if_exists="replace", index=False)
    table.to_csv(f"model/{name}.csv", index=False)
    print(f"{name}: {table.shape}")
con.close()

fact_order_items: (180519, 24)
dim_customer: (20652, 5)
dim_product: (118, 7)
dim_location: (3772, 6)
dim_shipping_mode: (4, 2)
dim_date: (1133, 10)
